# 🚀 Agoda Direct-API Crawler — chạy cả 3 Gate (1 cú nhấp)

Notebook này gọi lại code đã test trong `agoda_direct.py` (cùng thư mục `_direct_api/`).
Chạy lần lượt các cell từ trên xuống: **install → import → CONFIG → RUN ALL**.

- **Gate 0**: warm + bắt 1 request room-grid thật *(make-or-break)*
- **Gate 1**: replay verbatim + replay đổi ngày
- **Gate 2**: crawl thử nhỏ, đo tốc độ + độ chính xác → ra `DIRECT_<ngày>.csv`

⚠️ Phải chạy trên máy/IP của bạn (Akamai buộc cookie `_abck` theo TLS + IP).

In [1]:
# Cài 1 lần (bỏ qua nếu venv đã có sẵn)
!pip install -q curl_cffi playwright playwright-stealth pandas nest-asyncio
!playwright install chromium

source: Error encountered while sourcing file '/Users/hchinhtrung/.openclaw/completions/openclaw.fish':
source: No such file or directory


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
source: Error encountered while sourcing file '/Users/hchinhtrung/.openclaw/completions/openclaw.fish':
source: No such file or directory



In [2]:
import sys, os
from types import SimpleNamespace
from datetime import datetime, timedelta
import nest_asyncio; nest_asyncio.apply()

# Thư mục chứa agoda_direct.py (sửa nếu bạn đặt notebook nơi khác)
NB_DIR = '/Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/_direct_api'
if NB_DIR not in sys.path:
    sys.path.insert(0, NB_DIR)
os.chdir(NB_DIR)                      # để _capture/ và DIRECT_*.csv nằm trong _direct_api/

import importlib, agoda_direct as A
importlib.reload(A)                  # nạp lại nếu bạn vừa sửa agoda_direct.py
print('✅ import agoda_direct OK | warm profiles:', [imp for _, imp in A.WARM_PROFILES])

✅ import agoda_direct OK | warm profiles: ['chrome131', 'chrome124', 'chrome120', 'chrome116']


In [3]:
# ====== ⚙️ CONFIG — sửa ở đây ======
# CAPTURE_URL: 1 URL bất kỳ copy từ cột URL của CSV (hoặc từ trình duyệt khi mở 1 KS Agoda)
CAPTURE_URL = 'https://www.agoda.com/en-gb/may-hotel-saigon_2/hotel/ho-chi-minh-city-vn.html?currencyCode=VND&los=1&adults=2&rooms=1'
ROOM        = 'Narra Double'                 # tên phòng để sanity-check giá
INPUT_CSV   = '../agoda/agoda1/agoda1.csv'   # file input (đã có sẵn trong repo)
MAX_HOTELS  = 5                              # số KS crawl thử (Gate 2)
WEEKS       = 2                              # số tuần (Gate 2)

# Tuỳ chọn nâng cao (ghi đè config trong agoda_direct.py):
A.HEADLESS = True                            # đặt False để XEM trình duyệt chạy khi debug warm
# A.DAYS_PER_WEEK   = 3
# A.MAX_CONCURRENCY = 3
print('CONFIG:', CAPTURE_URL[:60], '... | room:', ROOM, '| input:', INPUT_CSV, '| max:', MAX_HOTELS, '| weeks:', WEEKS)

CONFIG: https://www.agoda.com/en-gb/may-hotel-saigon_2/hotel/ho-chi- ... | room: Narra Double | input: ../agoda/agoda1/agoda1.csv | max: 5 | weeks: 2


In [ ]:
# ====== ▶️ RUN ALL — chạy 3 gate tuần tự ======
async def run_all():
    bc = datetime.today().replace(hour=0, minute=0, second=0, microsecond=0) + timedelta(days=A.CHECKIN_OFFSET)
    print('='*64); print('🔥 GATE 0 — warm + capture (checkin =', bc.strftime('%Y-%m-%d'), ')'); print('='*64)
    cap = await A.warm_and_capture(CAPTURE_URL, bc, save=True)
    if cap.get('req') is None:
        print('\n⛔ Gate 0 FAIL: không bắt được request (đang bị chặn lúc warm).')
        print('   → thử lại / đổi mạng (bật 4G) / đặt A.HEADLESS = False rồi chạy lại cell CONFIG + cell này.')
        return
    if ROOM:
        print('  🔎 Sanity giá', repr(ROOM), '->', A.extract_from_agoda(cap.get('resp_json') or {}, ROOM))

    print('\n' + '='*64); print('🧪 GATE 1 — replay verbatim + đổi ngày'); print('='*64)
    await A.gate1_replay(SimpleNamespace(room=ROOM))

    print('\n' + '='*64); print('🚀 GATE 2 — direct-API crawl thử'); print('='*64)
    await A.gate2_crawl(SimpleNamespace(input=INPUT_CSV, max=MAX_HOTELS, weeks=WEEKS))

await run_all()

🔥 GATE 0 — warm + capture (checkin = 2026-07-01 )
  ✅ Bắt được room-grid (profile chrome131): POST https://www.agoda.com/api/v1/property/room-grid…
     body POST: CÓ | #cookies agoda: 35 | apiKey: nằm trong header/body
     response có rooms: True
     💾 Đã lưu /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/_direct_api/_capture/capture.json
  🔎 Sanity giá 'Narra Double' -> {'found': True, 'price': '2,824,074', 'room': 'Narra Double'}

🧪 GATE 1 — replay verbatim + đổi ngày
🧪 GATE 1a — replay VERBATIM (impersonate=chrome131, cùng ngày capture):
     status=200 | rooms=True | keys=['propertyName', 'propertyId', 'propertyType', 'cityId', 'countryId', 'searchCriteriaDescription']
     giá 'Narra Double': {'found': True, 'price': '2,824,074', 'room': 'Narra Double'}
🧪 GATE 1b — replay ĐỔI CHECKIN sang 2026-07-15:
     status=200 | rooms=False

  ── KẾT LUẬN GATE 1 ──
     1a verbatim : PASS ✅
     1b đổi ngày : FAIL ❌ → dùng warm theo từng KS (mặc định Gate 2)

🚀 

---
### (Tuỳ chọn) Chạy RIÊNG từng gate
Bỏ dấu `#` ở dòng tương ứng rồi chạy.

In [ ]:
# await A.gate0_capture(SimpleNamespace(url=CAPTURE_URL, room=ROOM))   # chỉ Gate 0
# await A.gate1_replay(SimpleNamespace(room=ROOM))                     # chỉ Gate 1
# await A.gate2_crawl(SimpleNamespace(input=INPUT_CSV, max=MAX_HOTELS, weeks=WEEKS))   # chỉ Gate 2